In [1]:
import os, ctypes

# Add the cu13 lib path so libnvrtc-builtins.so.13.0 can be found
nvrtc_path = "/home/tinkerspace/.local/lib/python3.10/site-packages/nvidia/cu13/lib"
os.environ["LD_LIBRARY_PATH"] = nvrtc_path + ":" + os.environ.get("LD_LIBRARY_PATH", "")

# Force-load the library right now so it's in memory before torch/ultralytics import it
ctypes.CDLL(os.path.join(nvrtc_path, "libnvrtc-builtins.so.13.0"))

print("✅ libnvrtc-builtins.so.13.0 loaded successfully!")

✅ libnvrtc-builtins.so.13.0 loaded successfully!


In [2]:
# Install ultralytics if not already present
!pip install -q ultralytics opencv-python tqdm

import os
import json
import cv2
import numpy as np
from tqdm import tqdm
from ultralytics import YOLO
import time
import datetime

# Verify your RTX 4000 SFF Ada is active
import torch
print(f"Using Device: {torch.cuda.get_device_name(0)}")

Using Device: NVIDIA RTX 4000 SFF Ada Generation


***Dont run the below cell***

In [3]:
# Setup paths pointing directly to Part 2
BASE_DATA_DIR = 'Data/idd20kII'
IMAGE_DIR = os.path.join(BASE_DATA_DIR, 'leftImg8bit')
MASK_DIR = os.path.join(BASE_DATA_DIR, 'gtFine')

TARGET_CLASS = "road"  

def convert_json_to_yolo(json_path, output_txt):
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)
            
        h = data["imgHeight"]
        w = data["imgWidth"]
        
        yolo_polygons = []
        for obj in data["objects"]:
            if obj["label"] == TARGET_CLASS:
                poly = obj["polygon"]
                # Flatten the [x, y] list and normalize coordinates
                norm_poly = []
                for point in poly:
                    norm_poly.append(str(point[0] / w)) # x
                    norm_poly.append(str(point[1] / h)) # y
                    
                yolo_polygons.append(f"0 {' '.join(norm_poly)}")
                
        if yolo_polygons:
            with open(output_txt, 'w') as f:
                f.write('\n'.join(yolo_polygons))
            return True
            
    except Exception as e:
        print(f"Error reading {json_path}: {e}")
        
    return False

# Convert both splits
for split in ['train', 'val']:
    split_mask_dir = os.path.join(MASK_DIR, split)
    print(f"Processing {split} JSON labels...")
    
    count = 0
    # Walk through the ground truth directory
    for root, _, files in os.walk(split_mask_dir):
        for file in files:
            if file.endswith('_polygons.json'):
                json_path = os.path.join(root, file)
                
                # Figure out the relative path to place the text file in the same image subfolder
                rel_path = os.path.relpath(root, split_mask_dir)
                target_img_dir = os.path.join(IMAGE_DIR, split, rel_path)
                
                # YOLO format requires the .txt to be identically named to the .jpg
                txt_name = file.replace('_gtFine_polygons.json', '_leftImg8bit.txt')
                output_txt_path = os.path.join(target_img_dir, txt_name)
                
                # If conversion is successful, count it
                if convert_json_to_yolo(json_path, output_txt_path):
                    count += 1
                    
    print(f"✅ Generated {count} YOLO labels for {split}.")


Processing train JSON labels...
✅ Generated 6830 YOLO labels for train.
Processing val JSON labels...
✅ Generated 1022 YOLO labels for val.


In [3]:
# Load the powerful X-Large segmentation model
model = YOLO('runs/segment/lane_project/v1_highres/weights/last.pt')
print("🚀 Starting Training on RTX 4000... calculating ETA...\n")
start_time = time.time()
# Train the model on your newly parsed labels
results = model.train(
    data='idd_lane.yaml',
    epochs=100,
    imgsz=1024,          # High resolution to catch thin lane lines (downsampled to memory limits)
    batch=8,             # Adjusted for 20GB RTX 4000 VRAM
    device=0,            # Target your RTX 4000
    workers=16,          # Utilize your CPU mapping
    project='lane_project',
    name='v1_highres',
    exist_ok=True,
    resume=True,
    amp=True             # Automatic Mixed Precision for RTX speed
)
# Calculate total runtime
total_seconds = time.time() - start_time
completion_time = str(datetime.timedelta(seconds=int(total_seconds)))
print(f"\n✅ Training Complete! Total time taken: {completion_time}\n")
print(f"Best weights saved at: {results.dir}/weights/best.pt")

🚀 Starting Training on RTX 4000... calculating ETA...

New https://pypi.org/project/ultralytics/8.4.21 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.19 🚀 Python-3.10.12 torch-2.10.0+cu130 CUDA:0 (NVIDIA RTX 4000 SFF Ada Generation, 20009MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=idd_lane.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs/segment/

/home/tinkerspace/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-09 15:28:25,235	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-03-09 15:28:25,302	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


WARNING ⚠️ ClearML installed but not initialized correctly, not logging this run. It seems ClearML is not configured on this machine!
To get started with ClearML, setup your own 'clearml-server' or create a free account at https://app.clear.ml
Setup instructions can be found here: https://clear.ml/docs

                   from  n    params  module                                       arguments                     
  0                  -1  1      2784  ultralytics.nn.modules.conv.Conv             [3, 96, 3, 2]                 
  1                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
  2                  -1  2    389760  ultralytics.nn.modules.block.C3k2            [192, 384, 2, True, 0.25]     
  3                  -1  1   1327872  ultralytics.nn.modules.conv.Conv             [384, 384, 3, 2]              
  4                  -1  2   1553664  ultralytics.nn.modules.block.C3k2            [384, 768, 2, True, 0.25]     
  5        

/home/tinkerspace/.local/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


WARNING ⚠️ CUDA out of memory with batch=8. Reducing to batch=4 and retrying (1/3).
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5105.2±3739.3 MB/s, size: 502.7 KB)
train: Scanning /home/tinkerspace/Lane-Detection/Data/idd20kII/leftImg8bit/train/201.cache... 6830 images, 204 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 7034/7034 2.3Git/s 0.0s
train: /home/tinkerspace/Lane-Detection/Data/idd20kII/leftImg8bit/train/213/frame57158_leftImg8bit.jpg: 1 duplicate labels removed
train: /home/tinkerspace/Lane-Detection/Data/idd20kII/leftImg8bit/train/224/frame2947_leftImg8bit.jpg: 1 duplicate labels removed
train: /home/tinkerspace/Lane-Detection/Data/idd20kII/leftImg8bit/train/235/frame10663_leftImg8bit.jpg: 1 duplicate labels removed
train: /home/tinkerspace/Lane-Detection/Data/idd20kII/leftImg8bit/train/238/frame1582_leftImg8bit.jpg: 1 duplicate labels removed
train: /home/tinkerspace/Lane-Detection/Data/idd20kII/leftImg8bit/train/302/frame5449_leftImg8bit.jpg: 1 duplicate labels removed

KeyboardInterrupt: 